# Chapter 03 제출 답안. 데이터의 첫인상 읽기

## 0. 제출 정보

- 이름: 이성민
- GitHub ID: minilee0307-dot
- 수정·재실행일: 2026-09-22
- 최종 제출 URL: https://github.com/minilee0307-dot/llm-data-analysis-study/blob/main/chapter03/chapter03.ipynb
- 실습 수행 상태: **COMPLETE** — 아래 모든 코드 셀을 순서대로 재실행하고 출력을 저장했다.
- GitHub 반영·표시 확인: **PARTIAL** — 수정본 업로드 및 원격 렌더링 확인 전이다.

공식 [실습 가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter03/chapter03.md)와 [제출 양식](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter03/templates/chapter03_assignment.md)의 1~6번에 맞춰 정리했다.
기존 실습에서 생성한 가상 쇼핑몰 CSV를 `data/`에 그대로 보존했다. 새로 생성하면 날짜가 달라질 수 있어 재생성하지 않았다. 실제 고객 정보는 아니다.
기존 ChatGPT 도움에 이어 이번 수정은 Codex의 코드 점검·실행·문장 정리 도움을 받았다. 모델이 제안한 내용과 실행으로 확인한 결과를 구분해 기록했다.

재실행: 이 저장소를 내려받고 Python 가상환경에 `chapter03/requirements.txt`를 설치한 뒤 해당 환경을 Notebook 커널로 선택하여 전체 실행한다. 원본 CSV는 수정하지 않는다.

## 1. 데이터 로딩과 구조 확인

### 실행/결과
파일 존재 여부, 파일별 행·열, 실제 컬럼명과 읽기 직후 dtype을 먼저 확인했다. 아래 표와 `head()`, `tail()`, `info()`가 실행 근거다.

In [1]:
from pathlib import Path
import sys, hashlib
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 60)

def find_chapter(start):
    for parent in [start.resolve(), *start.resolve().parents]:
        for candidate in [parent, parent / 'chapter03']:
            if (candidate / 'data' / 'customers.csv').is_file():
                return candidate
    raise FileNotFoundError('chapter03/data의 원본 CSV 4개를 확인하세요.')

CHAPTER = find_chapter(Path.cwd())
DATA_DIR = CHAPTER / 'data'
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('가상환경 사용:', sys.prefix != sys.base_prefix)
expected_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
file_check = pd.DataFrame([{'file': f, 'exists': (DATA_DIR/f).is_file(),
    'sha256': hashlib.sha256((DATA_DIR/f).read_bytes()).hexdigest()} for f in expected_files])
display(file_check)
datasets = {Path(f).stem: pd.read_csv(DATA_DIR/f) for f in expected_files}
customers, products, orders, order_items = [datasets[n] for n in ['customers','products','orders','order_items']]
shape_summary = pd.DataFrame([{'dataset': n, 'rows': len(d), 'columns': len(d.columns)} for n,d in datasets.items()])
display(shape_summary)
schema = pd.DataFrame([{'dataset': n, 'column': c, 'dtype_read': str(d[c].dtype)}
    for n,d in datasets.items() for c in d.columns])
display(schema)

Python: 3.14.3 | pandas: 3.0.5
가상환경 사용: True


,file,exists,sha256
0,customers.csv,True,cd53b7f52f960a13a67f509230f5a5ab6c68b3cdce2158...
1,products.csv,True,46ff06753ade3e71011ee08dd366148dcd9c36521f1118...
2,orders.csv,True,67cc403001e68134ff2fca73266a8f3b706aca05741c59...
3,order_items.csv,True,a7a5c3942b0886c66cb38cc6a10fdfa68dbcde90e86425...


,dataset,rows,columns
0,customers,150,6
1,products,100,4
2,orders,300,5
3,order_items,764,5


,dataset,column,dtype_read
0,customers,customer_id,int64
1,customers,name,str
2,customers,gender,str
3,customers,age,int64
4,customers,city,str
5,customers,signup_date,str
6,products,product_id,int64
7,products,product_name,str
8,products,category,str
9,products,price,int64


In [2]:
for name, df in datasets.items():
    print(name, 'head')
    display(df.head())
    df.info()
print('customers tail')
display(customers.tail())

customers head


,customer_id,name,gender,age,city,signup_date
0,1,김수민,F,19,광주,2024-05-11
1,2,장춘자,F,32,대구,2023-12-14
2,3,김상현,F,61,성남,2025-03-16
3,4,김재호,F,55,울산,2025-04-29
4,5,최준서,F,19,부산,2023-09-17


<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  150 non-null    int64
 1   name         150 non-null    str  
 2   gender       150 non-null    str  
 3   age          150 non-null    int64
 4   city         150 non-null    str  
 5   signup_date  150 non-null    str  
dtypes: int64(2), str(4)
memory usage: 7.2 KB
products head


,product_id,product_name,category,price
0,1,전자기기 상품 001,전자기기,160000
1,2,도서 상품 002,도서,34000
2,3,전자기기 상품 003,전자기기,152000
3,4,생활용품 상품 004,생활용품,70000
4,5,식품 상품 005,식품,186000


<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    100 non-null    int64
 1   product_name  100 non-null    str  
 2   category      100 non-null    str  
 3   price         100 non-null    int64
dtypes: int64(2), str(2)
memory usage: 3.3 KB
orders head


,order_id,customer_id,order_date,payment_method,order_status
0,1,123,2025-10-10,card,completed
1,2,77,2026-05-30,naver_pay,cancelled
2,3,138,2026-03-06,bank_transfer,cancelled
3,4,57,2026-06-13,kakao_pay,cancelled
4,5,125,2026-05-19,card,cancelled


<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        300 non-null    int64
 1   customer_id     300 non-null    int64
 2   order_date      300 non-null    str  
 3   payment_method  300 non-null    str  
 4   order_status    300 non-null    str  
dtypes: int64(2), str(3)
memory usage: 11.8 KB
order_items head


,order_item_id,order_id,product_id,quantity,unit_price
0,1,1,100,3,102000
1,2,1,87,5,25000
2,3,1,7,3,142000
3,4,1,9,3,193000
4,5,2,72,4,189000


<class 'pandas.DataFrame'>
RangeIndex: 764 entries, 0 to 763
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  764 non-null    int64
 1   order_id       764 non-null    int64
 2   product_id     764 non-null    int64
 3   quantity       764 non-null    int64
 4   unit_price     764 non-null    int64
dtypes: int64(5)
memory usage: 30.0 KB
customers tail


,customer_id,name,gender,age,city,signup_date
145,146,고준영,M,61,성남,2023-12-13
146,147,김예은,M,19,부산,2025-07-09
147,148,김준혁,M,29,고양,2024-03-27
148,149,안성현,M,20,부산,2026-05-01
149,150,이경숙,M,40,대전,2026-07-15


### Evidence
![데이터 규모 확인](images/step01_structure.png)

기존 실습 캡처이며, 이번 재실행의 `shape_summary`와 네 파일의 행·열 수가 모두 일치한다. 상세 컬럼과 dtype은 위 실행 표에 남겼다.

### 결과 관찰
customers는 150행 6열, products는 100행 4열, orders는 300행 5열, order_items는 764행 5열이다. 고객은 `customer_id, name, gender, age, city, signup_date`, 상품은 `product_id, product_name, category, price`, 주문은 `order_id, customer_id, order_date, payment_method, order_status`, 주문 상세는 `order_item_id, order_id, product_id, quantity, unit_price`로 구성되어 있다.
읽기 직후 ID와 `age, price, quantity, unit_price`는 `int64`, 나머지는 `str`이다. 날짜 두 컬럼도 처음에는 `str`였다.

### 나의 해석과 판단
주문 하나에 상품 여러 개가 들어갈 수 있으므로 주문 상세 764행을 주문 764건이라고 세면 안 된다. ID는 숫자형이어도 평균보다 고유성과 연결 여부가 중요하다고 판단했다.

### 업무·분석적 의미
매출 계산에는 주문 상세의 수량과 단가가 필요하고, 완료 주문만 선택하려면 주문 테이블의 상태를 연결해야 한다. 컬럼 이름만 보고 하나의 파일로 분석할 수 있다고 판단하면 계산 기준을 놓칠 수 있다.

### 한계와 추가 확인 사항
앞·뒤 몇 행은 전체 품질을 보장하지 않는다. 이 한계는 아래 전수 검사로 보완했다. 데이터는 가상 자료이므로 실제 고객 행동을 대표한다고 볼 수 없다.

## 2. 결측·중복·키 품질

### 실행/결과
전체 20개 원본 컬럼의 결측 개수·비율, 전체 행 중복, 네 고유키의 결측·중복을 별도로 검사했다.

In [3]:
key_columns = {'customers':'customer_id', 'products':'product_id',
               'orders':'order_id', 'order_items':'order_item_id'}
def check_data_overview(name, df, key):
    return {'dataset': name, 'missing': int(df.isna().sum().sum()),
            'duplicate_rows': int(df.duplicated().sum()), 'primary_key': key,
            'key_missing': int(df[key].isna().sum()),
            'key_duplicates': int(df[key].duplicated().sum()),
            'key_unique': int(df[key].nunique())}
quality = pd.DataFrame([check_data_overview(n, d, key_columns[n]) for n,d in datasets.items()])
display(quality)
missing_by_column = pd.DataFrame([{'dataset': n, 'column': c,
    'missing_count': int(d[c].isna().sum()), 'missing_rate_pct': d[c].isna().mean()*100}
    for n,d in datasets.items() for c in d.columns])
display(missing_by_column)
repeat_check = pd.DataFrame([
    {'check': 'order_items.order_id 반복 (첫 출현 제외)', 'rows': int(order_items.order_id.duplicated().sum())},
    {'check': '(order_id, product_id) 반복 (첫 출현 제외)', 'rows': int(order_items.duplicated(['order_id','product_id']).sum())}])
display(repeat_check)

,dataset,missing,duplicate_rows,primary_key,key_missing,key_duplicates,key_unique
0,customers,0,0,customer_id,0,0,150
1,products,0,0,product_id,0,0,100
2,orders,0,0,order_id,0,0,300
3,order_items,0,0,order_item_id,0,0,764


,dataset,column,missing_count,missing_rate_pct
0,customers,customer_id,0,0.0
1,customers,name,0,0.0
2,customers,gender,0,0.0
3,customers,age,0,0.0
4,customers,city,0,0.0
5,customers,signup_date,0,0.0
6,products,product_id,0,0.0
7,products,product_name,0,0.0
8,products,category,0,0.0
9,products,price,0,0.0


,check,rows
0,order_items.order_id 반복 (첫 출현 제외),464
1,"(order_id, product_id) 반복 (첫 출현 제외)",10


### Evidence
![결측·중복·고유 ID 점검](images/step02_quality.png)

기존 캡처의 0건 결과를 재실행으로 확인했다. 이번에는 `order_item_id`를 포함한 네 고유키의 결측·중복과 고유 개수를 위 표에 명시했다.

### 결과 관찰
20개 컬럼 모두 결측 0건(0%)이고 네 파일의 전체 행 중복도 각각 0건이다. `customers.customer_id`, `products.product_id`, `orders.order_id`, `order_items.order_item_id`는 각각 결측 0건·중복 0건이며 고유값 수는 150, 100, 300, 764개다.
반면 `order_items.order_id`의 첫 출현 이후 반복은 464행이다. `(order_id, product_id)` 조합도 첫 출현 이후 10행이 반복된다.

### 나의 해석과 판단
고유키 검사 범위에서는 우선 수정해야 할 문제가 없었다. 주문 ID 반복은 주문과 상세의 일대다 관계이므로 삭제하지 않았다. 같은 주문에서 같은 상품이 반복된 10행도 `order_item_id`는 서로 다르다. 원본 생성 코드는 상품을 반복 추출하므로 이 조합의 반복을 곧바로 입력 오류로 볼 수 없다.

### 업무·분석적 의미
고유키 중복은 병합 시 행과 금액을 늘릴 수 있지만, 정상 상세 행을 중복이라고 삭제하면 금액이 줄어든다. 중복을 처리하기 전에 한 행이 무엇을 의미하는지 확인해야 한다.

### 한계와 추가 확인 사항
결측·중복이 0이라는 사실은 날짜의 앞뒤 관계나 업무적 타당성까지 보장하지 않는다. 실제 거래에서는 같은 상품을 별도 상세 행으로 기록하는 규칙도 확인해야 한다.

## 3. 숫자형·범주형·날짜 점검

### 실행/결과
숫자형은 ID를 제외하고 통계를 구했다. 범주형은 일부만 자르지 않고 모든 값의 빈도를 확인했다. 날짜 변환 전 결측과 변환으로 새로 생긴 실패를 구분했다.

In [4]:
numeric = pd.concat([customers[['age']].describe().T,
    products[['price']].describe().T, order_items[['quantity','unit_price']].describe().T])
display(numeric)
for name, col in [('customers','gender'),('customers','city'),('products','category'),
                  ('orders','payment_method'),('orders','order_status')]:
    print(name + '.' + col)
    display(datasets[name][col].value_counts(dropna=False).rename('count').to_frame())
date_rows = []
for name, col in [('customers','signup_date'),('orders','order_date')]:
    raw = datasets[name][col]
    converted = pd.to_datetime(raw, format='%Y-%m-%d', errors='coerce')
    date_rows.append({'column': name+'.'+col, 'before': str(raw.dtype), 'after': str(converted.dtype),
        'missing_before': int(raw.isna().sum()),
        'conversion_failures': int((raw.notna() & converted.isna()).sum()),
        'min': str(converted.min().date()), 'max': str(converted.max().date())})
    datasets[name][col] = converted
date_summary = pd.DataFrame(date_rows)
display(date_summary)

,count,mean,std,min,25%,50%,75%,max
age,150.0,42.086667,15.613166,19.0,29.0,40.0,57.0,69.0
price,100.0,110040.000000,56433.910574,5000.0,65750.0,112000.0,161000.0,200000.0
quantity,764.0,3.053665,1.410873,1.0,2.0,3.0,4.0,5.0
unit_price,764.0,108561.518325,56996.770604,5000.0,62000.0,111000.0,161250.0,200000.0


customers.gender


,count
gender,
F,84
M,66


customers.city


,count
city,
성남,21
광주,17
부산,16
대구,15
서울,15
울산,14
인천,14
대전,14
수원,13


...

orders.payment_method


,count
payment_method,
kakao_pay,79
naver_pay,77
bank_transfer,74
card,70


orders.order_status


,count
order_status,
completed,184
cancelled,64
refunded,52


,column,before,after,missing_before,conversion_failures,min,max
0,customers.signup_date,str,datetime64[us],0,0,2023-09-16,2026-09-05
1,orders.order_date,str,datetime64[us],0,0,2025-09-09,2026-09-09


### Evidence
![기본 범위와 주문 날짜 확인](images/step03_distribution.png)

기존 캡처의 숫자 범위와 주문 기간이 이번 결과와 일치한다. 성별·결제수단을 포함한 전체 빈도와 가입일 변환 결과는 위 출력으로 보완했다.

### 결과 관찰
- age는 19~69세(평균 42.0867, 중앙값 40), price는 5,000~200,000(평균 110,040, 중앙값 112,000), quantity는 1~5(평균 3.0537, 중앙값 3), unit_price는 5,000~200,000(평균 108,561.5183, 중앙값 111,000)이다.
- 성별은 F 84명, M 66명이다. 지역은 성남 21, 광주 17, 부산 16, 대구·서울 각각 15, 울산·인천·대전 각각 14, 수원 13, 고양 11명이다.
- 상품 카테고리는 스포츠 19, 전자기기 17, 생활용품·뷰티 각각 16, 도서 14, 패션 11, 식품 7개다.
- 결제수단은 kakao_pay 79, naver_pay 77, bank_transfer 74, card 70건이다.
- 주문 상태는 **completed 184건, cancelled 64건, refunded 52건**으로 총 300건이다.
- 두 날짜는 `str`에서 `datetime64[us]`로 변환되었다. 변환 전 결측과 변환 실패는 모두 0건이다. 주문일은 **2025-09-09~2026-09-09**, 가입일은 **2023-09-16~2026-09-05**다.

### 나의 해석과 판단
숫자 범위에서는 음수 가격이나 0 이하 수량이 보이지 않았다. 그렇다고 통계적 이상치나 업무 오류가 전혀 없다고 단정하지는 않았다. cancelled와 refunded가 116건이므로 모든 주문 금액을 완료 주문 매출이라고 부르면 안 된다고 판단했다.

### 업무·분석적 의미
주문 상태를 거르지 않으면 분석 대상이 달라진다. 월별 비교에서도 시작 월과 끝 월은 한 달 전체가 아니므로 단순 합계를 같은 조건으로 비교하기 어렵다.

### 한계와 추가 확인 사항
통화 컬럼은 없어서 금액은 CSV의 단가 단위로 표시했다. 할인·세금·배송비·부분 환불 기록은 없어 실제 정산액을 계산할 수 없다. 날짜 변환 성공 여부와 날짜 사이의 논리적 순서는 별개의 문제라 다음 절에서 추가 검사했다.

## 4. CSV 간 키 관계 검증

### 실행/결과
세 FK가 부모 테이블에 존재하는지 확인한 뒤 `many_to_one` 조건으로 병합했다. 행 수 보존과 날짜 순서도 점검했다.

In [5]:
relationships = [('orders','customer_id','customers'), ('order_items','order_id','orders'),
                 ('order_items','product_id','products')]
relationship_check = pd.DataFrame([{'relationship': f'{child}.{key} -> {parent}.{key}',
    'invalid_rows': int((~datasets[child][key].isin(datasets[parent][key])).sum())}
    for child,key,parent in relationships])
display(relationship_check)
joined = order_items.merge(orders, on='order_id', how='left', validate='many_to_one')
joined = joined.merge(products, on='product_id', how='left', validate='many_to_one')
joined = joined.merge(customers[['customer_id','age','signup_date']], on='customer_id',
                      how='left', validate='many_to_one')
order_customer = orders.merge(customers[['customer_id','signup_date']], on='customer_id',
                              how='left', validate='many_to_one')
before_signup = order_customer.order_date < order_customer.signup_date
join_checks = pd.DataFrame([
    {'check':'병합 전 상세 행', 'result': len(order_items)},
    {'check':'병합 후 상세 행', 'result': len(joined)},
    {'check':'병합 후 category/age/order_status 결측 합계', 'result': int(joined[['category','age','order_status']].isna().sum().sum())},
    {'check':'상세가 없는 주문', 'result': int((~orders.order_id.isin(order_items.order_id)).sum())},
    {'check':'가입일보다 이른 주문 (주문 단위)', 'result': int(before_signup.sum())}])
display(join_checks)
display(order_customer.loc[before_signup, ['order_id','customer_id','order_date','signup_date']].head())
assert len(joined) == len(order_items), '병합 후 상세 행 수가 달라졌습니다.'

,relationship,invalid_rows
0,orders.customer_id -> customers.customer_id,0
1,order_items.order_id -> orders.order_id,0
2,order_items.product_id -> products.product_id,0


,check,result
0,병합 전 상세 행,764
1,병합 후 상세 행,764
2,병합 후 category/age/order_status 결측 합계,0
3,상세가 없는 주문,0
4,가입일보다 이른 주문 (주문 단위),48


,order_id,customer_id,order_date,signup_date
8,9,145,2025-10-13,2025-11-19
9,10,110,2025-11-29,2026-02-04
14,15,65,2026-03-22,2026-06-28
26,27,63,2025-11-15,2025-12-12
29,30,18,2026-06-01,2026-07-09


### Evidence
![세 FK 연결 확인](images/step04_relationship.png)

기존 캡처와 이번 FK 검사 결과가 일치한다. 추가한 병합·날짜 순서 검사는 위 `join_checks` 출력에 남겼다.

### 결과 관찰
없는 `customer_id` 0행, 없는 `order_id` 0행, 없는 `product_id` 0행이다. 병합 전후 상세 행은 모두 764행이고 연결 후 category·age·order_status의 결측도 0건이다. 상세가 없는 주문은 0건이다.
그러나 **가입일보다 이른 주문이 48건(300건 중 16%)** 있었다. 이는 상세 행 기준이 아닌 주문 기준으로 센 값이다.

### 나의 해석과 판단
키가 모두 연결된다는 것과 업무적으로 날짜가 자연스럽다는 것은 달랐다. 생성 코드를 보면 가입일과 주문일을 독립적으로 뽑고 있어 날짜 역전이 생길 수 있다. 다만 비회원 주문 등 실제 업무 규칙은 이 자료만으로 알 수 없으므로 48건을 임의 삭제하거나 날짜를 고치지 않았다.

### 업무·분석적 의미
FK 누락과 병합 중 행 증가를 확인하면 금액 집계가 빠지거나 부풀려지는 것을 막을 수 있다. 가입 시점을 기준으로 재구매·가입 후 경과일을 분석할 때에는 별도로 발견한 날짜 역전이 문제가 된다.

### 한계와 추가 확인 사항
현재 결과는 이 CSV 스냅샷에 대한 검사다. 실제 데이터에서 FK 누락이 생기면 누락 파일, 자료형, 수집 시점 등을 먼저 확인해야 하며 바로 행을 버리면 안 된다. 가입 이후 행동 분석은 날짜 생성·기록 기준을 확정한 뒤 진행해야 한다.

## 5. LLM 구조 설명 검증

### LLM에 제공한 Safe Context
기존 Notebook에는 구조 요약 프롬프트와 연령대별 매출 제안에 대한 검토가 저장되어 있었다. 다만 독립된 LLM 원문 응답은 확인할 수 없어 아래에서 기존 기록과 이번 Codex 보완 제안을 구분했다. 아래 프롬프트는 이번 실제 출력으로 갱신한 기록이며, 별도 API 호출을 실행했다고 주장하지 않는다. 고객 이름이나 개별 거래 행은 포함하지 않았다.

In [6]:
safe_context = '\n'.join([
    '온라인 쇼핑몰 가상 데이터의 구조를 검토해 주세요.',
    shape_summary.to_string(index=False), schema.to_string(index=False), quality.to_string(index=False),
    date_summary.to_string(index=False), relationship_check.to_string(index=False),
    'order_status 빈도: ' + str(orders.order_status.value_counts().to_dict()),
    '요청: 가능한 분석과 추가 점검을 제안하고, 확인된 사실과 아직 모르는 내용을 구분해 주세요.'
])
print(safe_context)

온라인 쇼핑몰 가상 데이터의 구조를 검토해 주세요.
    dataset  rows  columns
  customers   150        6
   products   100        4
     orders   300        5
order_items   764        5
    dataset         column dtype_read
  customers    customer_id      int64
  customers           name        str
  customers         gender        str
  customers            age      int64
  customers           city        str
  customers    signup_date        str
   products     product_id      int64
   products   product_name        str
   products       category        str
   products          price      int64
     orders       order_id      int64
     orders    customer_id      int64
     orders     order_date        str
     orders payment_method        str
     orders   order_status        str
order_items  order_item_id      int64
order_items       order_id      int64
order_items     product_id      int64
order_items       quantity      int64
order_items     unit_price      int64
    dataset  missing  duplicate_rows  

### 제안·실제 검증·판단 (Prompt Log)

| 출처 / 제안 | 실제 검증 결과 | 판단과 반영 |
| --- | --- | --- |
| 기존 Notebook 기록: age로 연령대별 매출을 바로 분석 | age 존재, 결측 0, 범위 19~69. 세 FK 누락 0. 그러나 상태는 completed 184 / cancelled 64 / refunded 52건 | **수정**. 기간·매출 산식·상태·연령 구간을 아래처럼 정하고 실행했다. |
| 이번 Codex 보완: 고유키와 FK를 따로 검사 | 네 PK 결측·중복 각 0, 상세 order_id 반복 464행 | **사용**. PK 검사를 완성하고 정상 FK 반복은 유지했다. |
| 이번 Codex 보완: 날짜 변환뿐 아니라 가입일과 주문일 순서도 검사 | 변환 실패는 0이지만 가입 전 주문은 48건 | **사용**. 날짜 역전을 품질 한계에 기록하고 원본을 보존했다. |
| 날짜만으로 가입 후 구매 행동을 해석할 수 있다는 가정 | 가입 전 주문 48건, 실제 가입·주문 기록 규칙 없음 | **보류**. 이는 추가 검토 가정이며 기존 LLM의 실제 발언으로 기록하지 않았다. 가입 시점 기반 해석은 하지 않았다. |

### 수정한 최종 분석 질문
**2025-09-09부터 2026-09-09까지(양 끝 날짜 포함), `order_status == 'completed'`인 주문만 대상으로 `quantity × unit_price`를 합산한 주문 금액은 고객의 기록된 연령대별로 어떻게 다른가?**
연령대는 10대 10~19세, 20대 20~29세, …, 60대 60~69세로 정의한다. 나이는 고객 테이블에 기록된 값이며 주문 당시 나이라고 단정하지 않는다. 여기서 매출은 위 산식으로 정의한 완료 주문 금액이며, 실제 순매출·이익을 뜻하지 않는다.

In [7]:
start_date, end_date = pd.Timestamp('2025-09-09'), pd.Timestamp('2026-09-09')
joined['line_amount'] = joined.quantity * joined.unit_price
in_period = joined.order_date.between(start_date, end_date, inclusive='both')
period_lines = joined.loc[in_period].copy()
status_amount = period_lines.groupby('order_status').agg(
    orders=('order_id','nunique'), lines=('order_item_id','size'), amount=('line_amount','sum'))
display(status_amount)
completed = period_lines.loc[period_lines.order_status.eq('completed')].copy()
completed['age_group'] = (completed.age // 10 * 10).astype(str) + '대'
age_sales = completed.groupby('age_group').agg(
    customers=('customer_id','nunique'), orders=('order_id','nunique'),
    lines=('order_item_id','size'), completed_amount=('line_amount','sum'))
display(age_sales)
category_check = period_lines.groupby('category').line_amount.sum().rename('all_status_amount').to_frame()
category_check['completed_amount'] = completed.groupby('category').line_amount.sum()
display(category_check)
# 병합 전의 상세 자료와 완료 주문 ID만으로 독립적으로 금액을 재검산한다.
selected_ids = orders.loc[orders.order_date.between(start_date, end_date) & orders.order_status.eq('completed'), 'order_id']
original_selected = order_items.loc[order_items.order_id.isin(selected_ids)]
independent_total = (original_selected.quantity * original_selected.unit_price).sum()
assert age_sales.completed_amount.sum() == independent_total
assert category_check.completed_amount.sum() == independent_total
print('대상 주문:', completed.order_id.nunique(), '| 상세:', len(completed))
print('완료 주문 합계:', int(independent_total), '| 연령대·카테고리 합계 대조: 일치')
print('전체 상태 주문 금액:', int(period_lines.line_amount.sum()))

,orders,lines,amount
order_status,,,
cancelled,64,162,62181000
completed,184,474,148990000
refunded,52,128,44439000


,customers,orders,lines,completed_amount
age_group,,,,
10대,2,4,11,4994000
20대,23,44,110,35906000
30대,25,50,133,37992000
40대,14,22,60,18677000
50대,14,24,57,19242000
60대,22,40,103,32179000


,all_status_amount,completed_amount
category,,
도서,24645000,16389000
뷰티,47551000,23383000
생활용품,34839000,23915000
스포츠,50174000,31743000
식품,33597000,16573000
전자기기,41003000,26400000
패션,23801000,10587000


대상 주문: 184 | 상세: 474
완료 주문 합계: 148990000 | 연령대·카테고리 합계 대조: 일치
전체 상태 주문 금액: 255610000


### Evidence
![최종 기준으로 계산한 검증 결과](images/step05_verified.png)

이번 재실행의 상태별·연령대별·카테고리별 출력 화면이다. 기존 `step05_llm.png`는 기준을 아직 정하기 전 기록이라 최종 Evidence에서는 교체했다.

### 결과 관찰
완료 주문은 184건, 상세 474행이며 합계는 **148,990,000**이다. 취소 주문 금액은 62,181,000, 환불 주문 금액은 44,439,000이고, 모든 상태를 합친 주문 금액은 **255,610,000**이다. 연령대별 및 카테고리별 완료 금액의 합계는 병합 전 원본으로 재계산한 148,990,000과 일치했다.

### 나의 해석과 판단
가장 유용한 점은 LLM의 아이디어를 실제 컬럼·키·조건으로 나누어 확인한 것이다. 반면 `age`가 있다는 사실만으로 바로 매출 분석이 준비되었다고 판단하는 것은 조심해야 한다. 기존 카테고리 집계는 모든 상태를 포함한 주문 금액이므로 완료 매출과 분리했다.

### 한계와 추가 확인 사항
연령대 합계 차이는 고객 수·주문 수의 차이도 반영한다. 이 결과만으로 특정 연령대의 개인 구매력이 높다거나 나이가 매출의 원인이라고 말할 수 없다. 가상 자료의 집계 연습이며, 실제 정산이나 시장 해석에 사용하지 않는다.

## 6. Chapter 03 최종 판단

### 데이터의 첫인상 3가지
1. 네 CSV는 총 1,314행으로 고객·상품·주문·상세 역할이 나뉜다. 병합 후에도 상세 764행이 보존되었다.
2. 결측·전체 중복·고유키 중복·FK 누락은 모두 0이지만, 가입 전 주문 48건은 따로 발견되었다. 기본 검사에서 문제가 없어도 모든 데이터가 타당한 것은 아니었다.
3. 매출 조건에 따라 전체 주문 금액 255,610,000과 완료 주문 금액 148,990,000으로 결과가 달라졌다. 실제 분석 질문에 조건을 적는 것이 중요했다.

### 다음 Chapter 전에 반드시 확인/처리해야 할 항목
1. **처리 완료:** 이번 집계의 기간, completed 조건, 수량×단가 산식과 연령 구간을 확정했다. 다음 장에서도 이 기준을 재사용한다.
2. **발견·보류:** 가입 전 주문 48건을 보존하고, 가입 시점을 사용하는 분석 전에는 생성·기록 규칙을 확인한다. 임의로 날짜를 고치지 않는다.
3. **추가 자료 필요:** 실제 매출로 확장하려면 통화·할인·세금·부분 환불과 나이의 기준 시점을 확인한다. 현재 자료에 없는 값을 채워 넣지 않는다.

### 현재 데이터만으로 단정할 수 없는 것
실제 고객 전체의 구매 성향, 연령과 매출의 인과관계, 실제 순매출·이익, 날짜 역전의 실제 업무 원인은 단정할 수 없다. 이번 장에서는 구조와 품질을 직접 검증하고 분석 가능한 범위까지 확인했다.

## 최종 제출 체크

- [x] Notebook을 처음부터 끝까지 실행했습니다.
- [x] 오류 셀이 남아 있지 않습니다.
- [x] 핵심 Evidence 5장을 본문에 연결했습니다.
- [x] 관찰과 해석을 구분했습니다.
- [x] 원본은 수업용 가상 데이터이며 개인정보/Secret을 추가하지 않았습니다.
- [ ] 수정한 `chapter03/chapter03.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 명시했습니다.
- [ ] eTL 제출/재제출 여부는 별도로 확인해야 합니다.